# Detección de Cáncer de Sangre (ALL) con Deep Learning en Google Colab

**Autor**: Equipo de Desarrollo  
**Institución**: Proyecto de Aprendizaje Profundo  
**Fecha**: Mayo 2026  
**Entorno**: Google Colab + PyTorch  

Este notebook implementa el modelo CNN para clasificación de células sanguíneas malignas vs benignas.

## 1. INSTALACIÓN DE DEPENDENCIAS

In [ ]:
# Verificar GPU disponible
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"Dispositivo: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Versión PyTorch: {torch.__version__}")

In [ ]:
# Instalar librerías necesarias
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn matplotlib seaborn pandas pillow
print("✅ Dependencias instaladas correctamente")

## 2. IMPORTAR LIBRERÍAS

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Configuración
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Usando dispositivo: {DEVICE}")

## 3. DESCARGAR Y PREPARAR DATASET

Opción 1: Descargar desde GitHub (recomendado)  
Opción 2: Subir manualmente via Colab

In [ ]:
# Montar Google Drive (si quieres acceder a archivos)
from google.colab import drive
drive.mount('/content/gdrive')
print("✅ Google Drive montado")

In [ ]:
# Clonar repositorio GitHub
!git clone https://github.com/Mercodn/BloodCancer.git
%cd BloodCancer
!ls -la

In [ ]:
# Crear directorios necesarios
Path("splits").mkdir(exist_ok=True)
Path("data_dictionary").mkdir(exist_ok=True)
Path("evaluation_results").mkdir(exist_ok=True)
print("✅ Directorios creados")

## 4. CREAR DATASET Y DIVIDIR EN TRAIN/VAL/TEST

In [ ]:
def collect_image_paths(base_dir="Blood cell Cancer [ALL]"):
    """Recolecta rutas de todas las imágenes del dataset"""
    images = []
    
    for category_path in Path(base_dir).iterdir():
        if category_path.is_dir():
            category = category_path.name
            label = "benign" if "Benign" in category else "malignant"
            
            for img_path in category_path.glob("*.jpg"):
                images.append((str(img_path), label))
    
    return images

# Recolectar imágenes
print("📂 Recolectando imágenes...")
images = collect_image_paths()
print(f"✅ Total de imágenes encontradas: {len(images)}")

# Mostrar distribución
from collections import Counter
labels_count = Counter([label for _, label in images])
for label, count in labels_count.items():
    print(f"   {label}: {count} imágenes ({count/len(images)*100:.1f}%)")

In [ ]:
import random

# Dividir en train/val/test (70/15/15)
random.seed(42)
random.shuffle(images)

total = len(images)
train_size = int(total * 0.7)
val_size = int(total * 0.15)

train_images = images[:train_size]
val_images = images[train_size:train_size + val_size]
test_images = images[train_size + val_size:]

print(f"🔀 División del dataset:")
print(f"   Train: {len(train_images)} ({len(train_images)/total*100:.1f}%)")
print(f"   Val:   {len(val_images)} ({len(val_images)/total*100:.1f}%)")
print(f"   Test:  {len(test_images)} ({len(test_images)/total*100:.1f}%)")

In [ ]:
# Guardar en CSV
import csv

def save_split_csv(images, filename):
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['image_path', 'label'])
        for img_path, label in images:
            writer.writerow([img_path, label])

save_split_csv(train_images, "splits/train.csv")
save_split_csv(val_images, "splits/val.csv")
save_split_csv(test_images, "splits/test.csv")

print("✅ CSV guardados:")
print("   - splits/train.csv")
print("   - splits/val.csv")
print("   - splits/test.csv")

## 5. CREAR CLASE DATASET

In [ ]:
class BloodCellDataset(Dataset):
    """Dataset personalizado para imágenes de células sanguíneas"""
    
    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform
        self.label_map = {"benign": 0, "malignant": 1}
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = Path(self.data.iloc[idx]["image_path"])
        label_str = self.data.iloc[idx]["label"]
        label = self.label_map[label_str]
        
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            raise FileNotFoundError(f"No se encontró: {img_path}")
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

print("✅ Clase BloodCellDataset creada")

## 6. DEFINIR ARQUITECTURA CNN

In [ ]:
class SimpleCNN(nn.Module):
    """CNN simple para clasificación binaria de células sanguíneas"""
    
    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()
        
        # Capas convolucionales
        self.conv_layers = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),
            
            # Bloque 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),
            
            # Bloque 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),
            
            # Bloque 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        
        # Capas fully connected
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

print("✅ Arquitectura SimpleCNN definida")

In [ ]:
# Contar parámetros del modelo
model_temp = SimpleCNN()
total_params = sum(p.numel() for p in model_temp.parameters())
trainable_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)

print(f"📊 Parámetros del modelo:")
print(f"   Total: {total_params:,}")
print(f"   Entrenables: {trainable_params:,}")

## 7. CONFIGURAR DATOS Y ENTRENAMIENTO

In [ ]:
# Configuración
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
IMAGE_SIZE = 224

# Transformaciones
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transformaciones configuradas")

In [ ]:
# Crear datasets
print("📂 Cargando datasets...")
train_dataset = BloodCellDataset("splits/train.csv", transform=train_transform)
val_dataset = BloodCellDataset("splits/val.csv", transform=val_transform)
test_dataset = BloodCellDataset("splits/test.csv", transform=val_transform)

# Crear dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Train: {len(train_dataset)} imágenes")
print(f"✅ Validation: {len(val_dataset)} imágenes")
print(f"✅ Test: {len(test_dataset)} imágenes")

## 8. FUNCIONES DE ENTRENAMIENTO

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Entrena un epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    """Valida el modelo"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

print("✅ Funciones de entrenamiento definidas")

## 9. ENTRENAR MODELO

In [ ]:
# Inicializar modelo
model = SimpleCNN(num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Usando dispositivo: {DEVICE}")
print(f"\n🚀 Comenzando entrenamiento...")
print("-" * 40)

# Historial
train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_acc = 0

# Entrenamiento
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    # Guardar mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "evaluation_results/best_model.pth")
        print(f"  ✓ Mejor modelo guardado (Acc: {val_acc:.2f}%)")
    print()

print("🏁 Entrenamiento completado!")

## 10. VISUALIZAR HISTORIAL DE ENTRENAMIENTO

In [ ]:
# Graficar historial
epochs = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Pérdida
ax1.plot(epochs, train_losses, 'b-', label='Train Loss')
ax1.plot(epochs, val_losses, 'r-', label='Val Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.set_title('Pérdida durante entrenamiento')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, train_accs, 'b-', label='Train Acc')
ax2.plot(epochs, val_accs, 'r-', label='Val Acc')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy durante entrenamiento')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Gráfica guardada: training_history.png")

## 11. EVALUACIÓN EN TEST SET

In [ ]:
def get_predictions(model, dataloader, device):
    """Obtiene predicciones del modelo"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probabilities[:, 1].cpu().numpy())
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# Cargar mejor modelo
model.load_state_dict(torch.load('evaluation_results/best_model.pth'))

print("📊 Evaluando modelo en conjunto de test...")
y_true, y_pred, y_prob = get_predictions(model, test_loader, DEVICE)

# Calcular métricas
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='binary')
recall = recall_score(y_true, y_pred, average='binary')
f1 = f1_score(y_true, y_pred, average='binary')
auc = roc_auc_score(y_true, y_prob)

print("\n" + "="*50)
print("RESUMEN FINAL - MÉTRICAS EN TEST")
print("="*50)
print(f"🎯 Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"🎯 Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"🎯 Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"🎯 F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"🎯 AUC-ROC:   {auc:.4f}")
print("="*50)

## 12. MATRIZ DE CONFUSIÓN

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benigno', 'Maligno'],
            yticklabels=['Benigno', 'Maligno'])
plt.title('Matriz de Confusión')
plt.ylabel('Valor Real')
plt.xlabel('Predicción')
plt.tight_layout()
plt.savefig('evaluation_results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Matriz de confusión guardada")

## 13. CURVA ROC

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc_score = roc_auc_score(y_true, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', label=f'ROC curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Curva ROC')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('evaluation_results/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Curva ROC guardada")

## 14. REPORTE DETALLADO

In [ ]:
# Reporte de clasificación
class_report = classification_report(y_true, y_pred, 
                                    target_names=['Benigno', 'Maligno'])

print("\n📋 REPORTE DE CLASIFICACIÓN DETALLADO")
print("-" * 50)
print(class_report)

## 15. GUARDAR MÉTRICAS EN ARCHIVO

In [ ]:
# Guardar reporte completo
with open('evaluation_results/metrics_report.txt', 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("REPORTE DE EVALUACIÓN DEL MODELO\n")
    f.write("="*60 + "\n\n")
    
    f.write("📊 MÉTRICAS PRINCIPALES\n")
    f.write("-"*40 + "\n")
    f.write(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)\n")
    f.write(f"Precision: {precision:.4f} ({precision*100:.2f}%)\n")
    f.write(f"Recall:    {recall:.4f} ({recall*100:.2f}%)\n")
    f.write(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)\n")
    f.write(f"AUC-ROC:   {auc:.4f}\n\n")
    
    f.write("📋 REPORTE DE CLASIFICACIÓN\n")
    f.write("-"*40 + "\n")
    f.write(class_report + "\n")
    
    f.write("🔢 MATRIZ DE CONFUSIÓN\n")
    f.write("-"*40 + "\n")
    f.write("                Predicho\n")
    f.write("               Benigno  Maligno\n")
    f.write(f"Real Benigno     {cm[0,0]:5d}   {cm[0,1]:5d}\n")
    f.write(f"Real Maligno     {cm[1,0]:5d}   {cm[1,1]:5d}\n\n")

print("✅ Reporte guardado: metrics_report.txt")

## 16. RESUMEN FINAL

In [ ]:
print("\n" + "="*60)
print("🎉 PROYECTO COMPLETADO EXITOSAMENTE")
print("="*60)
print(f"\n📊 Resultados Finales:")
print(f"   ✅ Accuracy:  {accuracy*100:.2f}%")
print(f"   ✅ Precision: {precision*100:.2f}%")
print(f"   ✅ Recall:    {recall*100:.2f}%")
print(f"   ✅ F1-Score:  {f1*100:.2f}%")
print(f"   ✅ AUC-ROC:   {auc:.4f}")

print(f"\n📁 Archivos Generados:")
print(f"   ✅ best_model.pth")
print(f"   ✅ metrics_report.txt")
print(f"   ✅ confusion_matrix.png")
print(f"   ✅ roc_curve.png")
print(f"   ✅ training_history.png")
print("\n" + "="*60)

---

## NOTAS IMPORTANTES

1. **Descarga de Archivos**: Usa el panel de archivos de Colab para descargar los resultados
2. **GPU**: Este notebook aprovecha GPU si está disponible, lo que acelera 10x el entrenamiento
3. **Reproducibilidad**: Los random seeds aseguran reproducibilidad
4. **Transferencia a Local**: Puedes descargar el modelo entrenado y usarlo en tu computadora

**Creado**: Mayo 2026  
**Repositorio**: https://github.com/Mercodn/BloodCancer